In [7]:
import torch
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

import pandas as pd
train_df= pd.read_csv('train_data.csv')
val_df= pd.read_csv('val_data.csv')

label_list=['hotel-general', 'bathroom-general', 'hotel-atmosphere', 'bathroom-atmosphere',
    'hotel-cleanliness', 'bathroom-cleanliness', 'hotel-facilities', 'bathroom-facilities',
    'hotel-location', 'bed-general', 'hotel-price', 'bed-cleanliness',
    'room-general', 'catering-general', 'room-atmosphere', 'catering-price',
    'room-cleanliness', 'parking', 'room-facilities', 'staff']

print(f'Train: {len(train_df)}')
print(f'Val: {len(val_df)}')

Device: cuda
Train: 3289
Val: 377


In [8]:
import numpy as np

total= len(train_df)
pos= train_df[label_list].sum().values
neg= total-pos

pos_weights= neg/pos
pos_weight_tensor= torch.tensor(pos_weights, dtype= torch.float).to(device)

print(pos_weight_tensor)

tensor([12.3699, 18.1221, 10.0000, 17.0714, 17.3743, 14.8889, 14.0183, 18.4615,
         5.5912, 13.1767, 18.3471, 17.1713, 14.9660, 10.7464,  7.5651, 17.5819,
        15.8667, 16.1302,  9.7134,  4.5840], device='cuda:0')


In [9]:
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts= df['text'].astype(str).tolist()
        self.labels= df[label_list].values.astype(np.float32)
        self.tokenizer= tokenizer
        self.max_length= max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding= self.tokenizer(
            self.texts[idx],
            add_special_tokens= True,
            max_length= self.max_length,
            padding='max_length',
            truncation= True,
            return_tensors= 'pt'
        )
        return{
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

model_name= 'xlm-roberta-large'
tokenizer= AutoTokenizer.from_pretrained(model_name)
model= AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_list), problem_type='multi_label_classification').to(device)

train_dataset= ReviewDataset(train_df, tokenizer)
train_loader= DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

val_dataset= ReviewDataset(val_df, tokenizer)
val_loader= DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch.nn as nn

EPOCHS= 10
LR= 2e-5

no_decay= ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters= [
    {'params': [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

optimizer= AdamW(optimizer_grouped_parameters, lr=LR)

total_steps= len(train_loader)*EPOCHS
warmup_steps= total_steps//10

scheduler= get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

criterion= nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

In [12]:
from sklearn.metrics import f1_score

def find_best_thresholds (y_true, y_prob):
    best_thresholds = []
    label_f1s= []

    for i, label in enumerate(label_list):
        best_t, best_f1= 0.5, 0.0
        for t in np.arange(0.1, 0.91, 0.05):
            preds= (y_prob[:, i]>= t).astype(int)
            f1= f1_score(y_true[:, i], preds, zero_division= 0)
            if f1> best_f1:
                best_f1, best_t= f1, t
        best_thresholds.append(best_t)
        label_f1s.append(best_f1)
        print(f' {label:<25} threshold= {best_t:.2f} F1={best_f1:.3f}')

    macro_f1= np.mean(label_f1s)
    print(f'\n Val Macro-F1: {macro_f1:.4f}')
    return best_thresholds, macro_f1

In [13]:
from tqdm import tqdm

best_val_f1= 0.0
best_thresholds= [0.5]* len(label_list)

for epoch in range(EPOCHS):
    model.train()
    total_loss= 0.0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids= batch['input_ids'].to(device)
        attention_mask= batch['attention_mask'].to(device)
        labels= batch['labels'].to(device)

        outputs= model(input_ids, attention_mask=attention_mask)
        loss= criterion(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss+= loss.item()
    avg_loss= total_loss/ len(train_loader)
    print(f'Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}')

    model.eval()
    all_probs= []
    all_labels= []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            input_ids= batch['input_ids'].to(device)
            attention_mask= batch['attention_mask'].to(device)
            outputs= model(input_ids, attention_mask=attention_mask)
            probs= torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(batch['labels'].cpu().numpy())

    all_probs= np.concatenate(all_probs, axis=0)
    all_labels= np.concatenate(all_labels, axis=0)

    epoch_thresholds, macro_f1= find_best_thresholds(all_labels, all_probs)

    if macro_f1> best_val_f1:
        best_val_f1= macro_f1
        best_thresholds= epoch_thresholds
        model.save_pretrained('best_xlm_roberta_model')
        tokenizer.save_pretrained('best_xlm_roberta_model')
        np.save('best_thresholds.npy', np.array(best_thresholds))
        print(f'New best macro-F1: {best_val_f1:.4f}')

100%|██████████| 412/412 [05:35<00:00,  1.23it/s]


Epoch 1 Loss: 1.1876


100%|██████████| 48/48 [00:10<00:00,  4.56it/s]


 hotel-general             threshold= 0.70 F1=0.381
 bathroom-general          threshold= 0.75 F1=0.279
 hotel-atmosphere          threshold= 0.65 F1=0.433
 bathroom-atmosphere       threshold= 0.85 F1=0.444
 hotel-cleanliness         threshold= 0.60 F1=0.632
 bathroom-cleanliness      threshold= 0.85 F1=0.368
 hotel-facilities          threshold= 0.60 F1=0.356
 bathroom-facilities       threshold= 0.75 F1=0.588
 hotel-location            threshold= 0.70 F1=0.660
 bed-general               threshold= 0.80 F1=0.667
 hotel-price               threshold= 0.70 F1=0.379
 bed-cleanliness           threshold= 0.45 F1=0.571
 room-general              threshold= 0.75 F1=0.298
 catering-general          threshold= 0.55 F1=0.606
 room-atmosphere           threshold= 0.55 F1=0.604
 catering-price            threshold= 0.55 F1=0.250
 room-cleanliness          threshold= 0.65 F1=0.471
 parking                   threshold= 0.65 F1=0.880
 room-facilities           threshold= 0.65 F1=0.583
 staff      

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 10 Loss: 0.0538


100%|██████████| 48/48 [00:10<00:00,  4.68it/s]


 hotel-general             threshold= 0.25 F1=0.598
 bathroom-general          threshold= 0.55 F1=0.476
 hotel-atmosphere          threshold= 0.85 F1=0.737
 bathroom-atmosphere       threshold= 0.35 F1=0.579
 hotel-cleanliness         threshold= 0.90 F1=0.741
 bathroom-cleanliness      threshold= 0.80 F1=0.667
 hotel-facilities          threshold= 0.30 F1=0.645
 bathroom-facilities       threshold= 0.90 F1=0.828
 hotel-location            threshold= 0.55 F1=0.955
 bed-general               threshold= 0.45 F1=0.930
 hotel-price               threshold= 0.55 F1=0.806
 bed-cleanliness           threshold= 0.80 F1=0.545
 room-general              threshold= 0.55 F1=0.679
 catering-general          threshold= 0.80 F1=0.919
 room-atmosphere           threshold= 0.55 F1=0.798
 catering-price            threshold= 0.65 F1=0.750
 room-cleanliness          threshold= 0.90 F1=0.778
 parking                   threshold= 0.90 F1=0.923
 room-facilities           threshold= 0.75 F1=0.869
 staff      

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best macro-F1: 0.7572
